In [ ]:
%pip install numpy trimesh
import numpy as np
import trimesh
import matplotlib.pyplot as plt
from scipy.spatial import Delaunay
import time
import tqdm as tqdm


# Poisson Equation in 2D
$$-\Delta_\Gamma u=f \quad \text{(Laplace Beltrami)}$$


## Weak formulation
$u\in C^{2}(\Omega) \rightarrow u \in C^1(\Omega)$

$$$$

$$-\int_\Gamma v \Delta_\Gamma u \hspace{1mm} dS=\int_\Gamma fv \hspace{1mm} dS$$

$$$$

$$\int_\Gamma \nabla_\Gamma u \cdot \nabla_\Gamma v \hspace{1mm} dS - \int_{\partial \Gamma} \frac{\partial u}{\partial \nu} v \hspace{1mm} ds =\int_\Omega fv \hspace{1mm} dS \quad \text{(Divergence Theorem)}$$

# Mesh Solution

In a mesh our domain is partitioned into quadrilateral meshes, where each vertex has its own $u$ value. Each point on the surface can then be approximated by an interpolation within the quadrilateral face (element)

## Basis Function

We will utilize a simple lagrange basis which utilizes a bicubic quadrilateral $(Q_3)$.


$$\Psi_i = \bigg\{ \begin{matrix} 1 & \text{node i} \\ 0 & \text{not node i}\end{matrix}$$
 
## Discretization

$$u_h(x,y)=\sum_{i}^NU_i\Psi_i(x,y)$$
$$$$
$$N = \text{total number of vertices}$$
$$U_i = \text{value of u at vertex i}$$
$$\Psi_i = \text{basis function of vertex i}$$

$$$$

 $u=u_h$ and $v=\sum_i \Psi_i \quad \text{(Galerkin Method)} $

$$$$
Ignoring boundary conditions, we get a system of equations involving a stiffness matrix (A) and a load vector (b)

$$$$

$$\int_\Gamma \left(\sum_i^N u_i\nabla_\Gamma \Psi_i\right) \left(\sum_j^N \nabla_\Gamma \Psi_j\right) d S =\int_\Gamma \sum_i^N \Psi_i f \hspace{1mm}
dS $$

$$$$

$$A_{i,j} = \int_{\Omega}\nabla_\Gamma \Psi_i \cdot \nabla_\Gamma \Psi_j \hspace{1mm} dS$$
$$b_{i} = \int_{\Gamma} \Psi_i f \hspace{1mm} dS$$

$$$$

$$AU=b, \quad U_i=u_i$$
$$$$
$$A\in \mathbb{R}^{N \times N},\quad b\in \mathbb{R}^{N},\quad U\in \mathbb{R}^{N}$$

# Elements

A bicubic element consistens of 16 nodes we can label $(i,j)$ each ranging from $[0,3]$

Within each element, each node has the following basis function:

$$\Psi_{i,j}(\xi,\eta) = \ell_i(\xi)\ell_j(\eta) $$

- $\ell_0(t) = -\frac{9}{16}\left(t+\frac{1}{3}\right)\left(t-\frac{1}{3}\right)\left(t-1\right) $
- $\ell_1(t) = \frac{27}{16}\left(t+1\right)\left(t-\frac{1}{3}\right)\left(t-1\right) $
- $\ell_2(t) = -\frac{27}{16}\left(t+1\right)\left(t+\frac{1}{3}\right)\left(t-1\right) $
- $\ell_3(t) = \frac{9}{16}\left(t+1\right)\left(t+\frac{1}{3}\right)\left(t-\frac{1}{3}\right) $
 



## Gradient

To determine the gradient at a point we can consider the gradient at the local coordinate system and then transform with a jacobian. (Covariant Basis)

$$J = \begin{bmatrix}
\mathbf{g}_1 & \mathbf{g}_2
\end{bmatrix}

=
\begin{bmatrix}
\frac{\partial x}{\partial \xi} & \frac{\partial x}{\partial \eta} \\
\frac{\partial y}{\partial \xi} & \frac{\partial y}{\partial \eta} \\
\frac{\partial z}{\partial \xi} & \frac{\partial z}{\partial \eta} \\
\end{bmatrix}
$$

Which can be computed explicity considering the basis functions

$$\mathbf{g}_1 = \sum_{i=1}^{16} \frac{\partial \Psi_i}{\partial \xi} \mathbf{x}_i \quad \mathbf{g}_2 = \sum_{i=1}^{16} \frac{\partial \Psi_i}{\partial \eta} \mathbf{x}_i$$

To make measurements we need to construct a metric tensor:

$$g=J^TJ = [g_{ij}] = \begin{bmatrix}

\mathbf{g}_1 \cdot \mathbf{g}_1 & \mathbf{g}_1 \cdot \mathbf{g}_2 \\
\mathbf{g}_2 \cdot \mathbf{g}_1 & \mathbf{g}_2 \cdot \mathbf{g}_2

\end{bmatrix}$$

The area element can be written as $dA = |\mathbf{g_1}\times \mathbf{g_2}| d\xi d\eta = \sqrt{\det(g)}d\xi d\eta$  

To project these derivatives back to physical space we need the inverse of the metric tensor (Contravariant Basis)

$$g^{-1} = [g^{ij}] = \frac{1}{\det(g)} \begin{bmatrix} g_{22} & -g_{12} \\ -g_{21} & g_{11}\end{bmatrix}$$

We can utilize this to compute the contravariant tangent vectors $\mathbf{g}^1$ and $\mathbf{g}^2$ These are dual vectors to the original tangent vectors ($\mathbf{g}^i \cdot \mathbf{g}_j = \delta_{ij}$, where $\delta_{ij}$ is the kronecker delta function)

$$\mathbf{g}^1 = g^{11} \mathbf{g}_1 + g^{12} \mathbf{g_2}$$
$$\mathbf{g}^2 = g^{21} \mathbf{g}_1 + g^{22} \mathbf{g_2}$$

Finally we can apply these contravariant dual vectors to the gradient of basis functions:

$$\nabla_\Gamma \Psi_i = \frac{\partial \Psi_i}{\partial \xi} \mathbf{g}^1 + \frac{\partial \Psi_i}{\partial \eta} \mathbf{g}^2$$

$$\nabla_\Gamma \Psi_i = \left(g^{11}\frac{\partial \Psi_i}{\partial \xi} + g^{12}\frac{\partial \Psi_i}{\partial \eta}\right)\mathbf{g}_1 +\left(g^{21}\frac{\partial \Psi_i}{\partial \xi} + g^{22}\frac{\partial \Psi_i}{\partial \eta}\right) \mathbf{g}_2$$

$$J^\dagger = g^{-1} J^T$$

$$\nabla_{\Gamma} \Psi_i = (J^\dagger)^T \begin{bmatrix} \frac{\partial N_i}{\partial \xi} \\ \frac{\partial N_i}{\partial \eta} \end{bmatrix}$$



In [ ]:
class Element:
    def __init__(self, nodes):
        self.nodes = nodes

    def basis(self,t):
       return [
            -9/16 * (t + 1/3) * (t - 1/3) * (t - 1),
            27/16 * (t + 1) * (t - 1/3) * (t - 1),
            -27/16 * (t + 1) * (t + 1/3) * (t - 1),
            9/16 * (t + 1) * (t + 1/3) * (t - 1/3)
        ]
    
    def basis_derivative(self,t):
        return [
            -9/16 * (3*t**2 - 2*t - 1/9),
            27/16 * (3*t**2 - 2*t/3 - 1),
            -27/16 * (3*t**2 + 2*t/3 - 1),
            9/16 * (3*t**2 + 2*t - 1/9)
        ]
    
    def basis_functions(self, xi, eta):
        psi = np.zeros(16)
        for i in range(4):
            for j in range(4):
                index = i + 4 * j
                psi[index] = self.basis(xi)[i] * self.basis(eta)[j]
        return psi
    
    def basis_function_gradients(self, xi, eta):
        dpsi_dxi = np.zeros((16, 2))
        for i in range(4):
            for j in range(4):
                index = i + 4 * j
                dpsi_dxi[index, 0] = self.basis_derivative(xi)[i] * self.basis(eta)[j]
                dpsi_dxi[index, 1] = self.basis(xi)[i] * self.basis_derivative(eta)[j]
        return dpsi_dxi

    def compute_jacobian(self, xi, eta):
        dpsi_dxi = self.basis_function_gradients(xi, eta)
        return self.nodes.T @ dpsi_dxi

    def compute_covariant(self,xi, eta):
        jacobian = self.compute_jacobian(xi, eta)
        return jacobian.T @ jacobian

    def compute_contravariant(self,xi, eta):
        return np.linalg.inv(self.compute_covariant(xi, eta))
    

    def pseudoinverse(self,xi, eta):
        return self.compute_contravariant(xi, eta) @ self.compute_jacobian(xi, eta).T

    def compute_gradients(self,xi, eta):

        gradients = np.zeros((16, 3))
        dpsi_dxi = self.basis_function_gradients(xi, eta)
        pinv_T = self.pseudoinverse(xi, eta).T         
        
        for i in range(16):
            gradients[i] = pinv_T @ dpsi_dxi[i]
            
        return gradients
    
    def compute_gradient(self, xi, eta, i):
        dpsi_dxi = self.basis_function_gradients(xi, eta)
        pinv_T = self.pseudoinverse(xi, eta).T
        return pinv_T @ dpsi_dxi[i]
    

    def compute_area(self,xi, eta):
        return np.sqrt(np.linalg.det(self.compute_covariant(xi, eta)))

    def local_to_global(self, local_coords):
        xi, eta = local_coords
        return np.array(self.basis_functions(xi, eta)) @ self.nodes

    

#  Local Stiffness Matrix

$$A_{i,j}^e=\int_{\Gamma_e}\nabla_\Gamma \Psi_i \cdot \nabla_\Gamma \Psi_j  dS$$

This integral is nonlinear so we will evaluate it with a gaussian quadrature, we will use a 7 point gaussian quadrature.


$$A_{i,j}^e=\sum_{k=1}^7\nabla_\Gamma \Psi_i(\xi_k,\eta_k) \cdot \nabla_\Gamma \Psi_j(\xi_k,\eta_k) \cdot w_k \cdot E_a$$

$$A^e=\begin{bmatrix}
A_{0,0} & A_{0,1} & A_{0,2} & A_{0,3} \\
A_{1,0} & A_{1,1} & A_{1,2} & A_{1,3} \\
A_{2,0} & A_{2,1} & A_{2,2} & A_{2,3} \\
A_{3,0} & A_{3,1} & A_{3,2} & A_{3,3} \\
\end{bmatrix}$$

$$$$

Note that these indices are local, they each represent a global index from $0...N$

# Global Stiffness Matrix
for each vertex add all the local entries to their corresponding global entry



In [ ]:
from numpy.polynomial.legendre import leggauss
import numpy as np

def quadrature_points(n=5):
    pts, wts = leggauss(n)

    GP = []
    weights = []

    for i in range(n):
        for j in range(n):
            xi = pts[i]
            eta = pts[j]

            weight = wts[i] * wts[j]

            GP.append([xi, eta])
            weights.append(weight)

    return np.array(GP), np.array(weights)

def update_global_stiffness(A,Vertices,face):
  element = Element(Vertices[face])
  GP, weights = quadrature_points()
  for k in range(len(GP)):
    xi, eta = GP[k]
    weight = weights[k]
    grad = element.compute_gradients(xi, eta)
    area = element.compute_area(xi, eta)
    for i in range(16):
      for j in range(16):
        A[face[i], face[j]] += weight * grad[i].dot(grad[j]) * area


def create_global_stiffness(A,Vertices,Faces):
  with tqdm.tqdm(total=len(Faces), desc="Assembling global stiffness matrix") as pbar:
    for face in Faces:
      update_global_stiffness(A,Vertices,face)
      pbar.update(1)



# Local Load Vector

$$b_i^e = \int_{\Gamma_e} f \Psi_i \hspace{1mm} dS$$

## Quadrature

We will utilize the same quadrature as the stiffness matrix
$$$$
$$b_i^e=\sum_{k=1}^3 f(\mathbf{x}(\xi_k,\eta_k)) \cdot \Psi_i(\xi_k,\eta_k) \cdot w_k \cdot E_a$$

# Global Load Vector
for each vertex add all the local entries to their corresponding global entry

In [ ]:
def update_global_load(b,f,Vertices,face):
  element = Element(Vertices[face])
  GP, weights = quadrature_points()
  
  for k in range(len(GP)):
    xi, eta = GP[k]
    weight = weights[k]
    x,y,z = element.local_to_global((xi,eta))
    area = element.compute_area(xi, eta)
    base = element.basis_functions(xi, eta)
    for i in range(16):
      b[face[i]] += weight * f(x,y,z) * base[i] * area

def create_global_load(b,f,Vertices,Faces):
  with tqdm.tqdm(total=len(Faces), desc="Assembling global load vector") as pbar:
    for face in Faces:
      update_global_load(b,f,Vertices,face)
      pbar.update(1)



# Bounds

In [ ]:
def make_bicubic_mesh(Vertices, Quads, project=None):

    vertices_list = [np.array(v, dtype=float) for v in Vertices]

    edge_to_nodes = {}
    new_faces = []

    def maybe_project(p):
        if project is None:
            return p
        return np.array(project(p), dtype=float)

    def add_vertex(p):
        vertices_list.append(maybe_project(p))
        return len(vertices_list) - 1

    def get_edge_node(a, b, m):

        key = tuple(sorted((a, b)))

        if key not in edge_to_nodes:
            v0 = vertices_list[key[0]]
            v1 = vertices_list[key[1]]

            p1 = (2.0/3.0) * v0 + (1.0/3.0) * v1
            p2 = (1.0/3.0) * v0 + (2.0/3.0) * v1

            idx1 = add_vertex(p1)
            idx2 = add_vertex(p2)

            edge_to_nodes[key] = [idx1, idx2]

        nodes = edge_to_nodes[key]

        if (a, b) == key:
            return nodes[m - 1]
        else:
            return nodes[2 - m]

    for quad in Quads:
        q0, q1, q2, q3 = quad

        face = np.empty(16, dtype=int)

        # Corners
        face[0]  = q0
        face[3]  = q1
        face[15] = q2
        face[12] = q3

        # Bottom edge: q0 -> q1
        face[1] = get_edge_node(q0, q1, 1)
        face[2] = get_edge_node(q0, q1, 2)

        # Right edge: q1 -> q2
        face[7]  = get_edge_node(q1, q2, 1)
        face[11] = get_edge_node(q1, q2, 2)

        # Top edge: q3 -> q2
        face[13] = get_edge_node(q3, q2, 1)
        face[14] = get_edge_node(q3, q2, 2)

        # Left edge: q0 -> q3
        face[4] = get_edge_node(q0, q3, 1)
        face[8] = get_edge_node(q0, q3, 2)

        # Interior nodes
        X0 = vertices_list[q0]
        X1 = vertices_list[q1]
        X2 = vertices_list[q2]
        X3 = vertices_list[q3]

        for j in [1, 2]:
            for i in [1, 2]:
                s = i / 3.0
                t = j / 3.0

                p = (
                    (1 - s) * (1 - t) * X0
                    + s * (1 - t) * X1
                    + s * t * X2
                    + (1 - s) * t * X3
                )

                A = i + 4*j
                face[A] = add_vertex(p)

        new_faces.append(face)

    return np.array(vertices_list), np.array(new_faces)


def get_Boundary(Faces):

    boundary_edges = []
    edge_count = {}
    boundary_vertices = set()

    # Corner pairs associated with local bicubic edge nodes
    edge_data = [
        ((0, 3),   [0, 1, 2, 3]),       # bottom
        ((3, 15),  [3, 7, 11, 15]),     # right
        ((12, 15), [12, 13, 14, 15]),   # top
        ((0, 12),  [0, 4, 8, 12]),      # left
    ]

    # Step 1: Count structural corner edges
    for face in Faces:
        for corner_pair, local_edge_nodes in edge_data:
            a_local, b_local = corner_pair
            a = face[a_local]
            b = face[b_local]

            edge = tuple(sorted((a, b)))
            edge_count[edge] = edge_count.get(edge, 0) + 1

    # Step 2: Identify boundary edges
    for edge, count in edge_count.items():
        if count == 1:
            boundary_edges.append(edge)

    boundary_edge_set = set(boundary_edges)

    # Step 3: Add all bicubic edge nodes on boundary edges
    for face in Faces:
        for corner_pair, local_edge_nodes in edge_data:
            a_local, b_local = corner_pair
            a = face[a_local]
            b = face[b_local]

            edge = tuple(sorted((a, b)))

            if edge in boundary_edge_set:
                for local_node in local_edge_nodes:
                    boundary_vertices.add(face[local_node])

    return list(boundary_vertices), boundary_edges

# Dirichlet Boundary Condition

$$u=g \quad \text{on} \hspace{3mm}\Gamma_d$$

This means for every boundary ($d$), $U_d=g(p_d)=g_d$

We need to reset all the boundary values and apply the boundary value term in the weak formulation
$$$$

1. Zero out the row and column of boundary $A_{i,d} = A_{d,i}= 0$

2. Set diagonal entry to one $A_{d,d}=1$

3. set load vector entry to g $F_d=g_d$

$$$$

These steps ensure that $U_d = g_d$

In [ ]:
def apply_dirichlet(A, b, Vertices, boundary_vertices, g):
    for d in boundary_vertices:
        x, y, z = Vertices[d]
        gd = g(x, y, z)

        # adjust RHS for all equations
        b[:] -= A[:, d] * gd

        # zero row and column
        A[d, :] = 0.0
        A[:, d] = 0.0

        # impose value
        A[d, d] = 1.0
        b[d] = gd

## Simple Quad Mesh Creator 

In [ ]:
def make_structured_quad_mesh(xmin, xmax, ymin, ymax, nx, ny, surface=None):
    
    xs = np.linspace(xmin, xmax, nx + 1)
    ys = np.linspace(ymin, ymax, ny + 1)

    vertices = []

    for j in range(ny + 1):
        for i in range(nx + 1):
            x = xs[i]
            y = ys[j]

            if surface is None:
                z = 0.0
            else:
                z = surface(x, y)

            vertices.append([x, y, z])

    vertices = np.array(vertices)

    def vid(i, j):
        return j * (nx + 1) + i

    quads = []

    for j in range(ny):
        for i in range(nx):
            q0 = vid(i, j)
            q1 = vid(i + 1, j)
            q2 = vid(i + 1, j + 1)
            q3 = vid(i, j + 1)

            quads.append([q0, q1, q2, q3])

    return np.array(vertices), np.array(quads)


def subdivide_quad_mesh(vertices, quads, project=None):

    vertices = np.asarray(vertices, dtype=float)
    quads = np.asarray(quads, dtype=int)

    new_vertices = vertices.tolist()
    new_quads = []

    edge_midpoint_map = {}

    def maybe_project(p):
        if project is None:
            return p
        return np.asarray(project(p), dtype=float)

    def add_vertex(p):
        p = maybe_project(np.asarray(p, dtype=float))
        new_vertices.append(p.tolist())
        return len(new_vertices) - 1

    def get_edge_midpoint(va, vb):
        """
        Get or create midpoint for an original mesh edge.
        This ensures adjacent quads share the same midpoint node.
        """
        key = tuple(sorted((int(va), int(vb))))

        if key not in edge_midpoint_map:
            p = 0.5 * (vertices[key[0]] + vertices[key[1]])
            edge_midpoint_map[key] = add_vertex(p)

        return edge_midpoint_map[key]

    for quad in quads:
        v0, v1, v2, v3 = map(int, quad)

        # Shared edge midpoints
        m01 = get_edge_midpoint(v0, v1)
        m12 = get_edge_midpoint(v1, v2)
        m23 = get_edge_midpoint(v2, v3)
        m30 = get_edge_midpoint(v3, v0)

        # Face center: unique per quad
        center_point = 0.25 * (
            vertices[v0]
            + vertices[v1]
            + vertices[v2]
            + vertices[v3]
        )
        c = add_vertex(center_point)

        # Four child quads, preserving orientation
        new_quads.append([v0,  m01, c,   m30])
        new_quads.append([m01, v1,  m12, c])
        new_quads.append([c,   m12, v2,  m23])
        new_quads.append([m30, c,   m23, v3])

    return np.asarray(new_vertices, dtype=float), np.asarray(new_quads, dtype=int)

In [ ]:
vertices, quads = make_structured_quad_mesh(-1, 1, -1, 1, 3, 3)

print(len(vertices), "vertices")
for i in range(4):
    vertices, quads = subdivide_quad_mesh(vertices, quads)
    print(len(vertices), "vertices")

# Test

In [ ]:
def franke(x,y):
  return 0.75*np.exp(-(9*x-2)**2/4-(9*y-2)**2/4) + 0.75*np.exp(-(9*x+1)**2/49-(9*y+1)/10) + 0.5*np.exp(-(9*x-7)**2/4-(9*y-3)**2/4) - 0.2*np.exp(-(9*x-4)**2-(9*y-7)**2)

def lap_franke(x,y):
  term1 = 243/16 * ((9*x-2)**2 + (9*y-2)**2-4) * np.exp(-(9*x-2)**2/4-(9*y-2)**2/4)
  term2 = 243/960400 * (400*(9*x+1)**2 - 7399) * np.exp(-(9*x+1)**2/49-(9*y+1)/10)
  term3 = 81/8 * ((9*x-7)**2 + (9*y-3)**2-4) * np.exp(-(9*x-7)**2/4-(9*y-3)**2/4)
  term4 = 324/5 * (1-(9*x-4)**2-(9*y-7)**2) * np.exp(-(9*x-4)**2-(9*y-7)**2)
  return term1 + term2 + term3 + term4


def surface(X):
    return (X[0], X[1], X[0]**2 + X[1]**2)
  #return (X[0], X[1], 1/np.pi * np.sin(2*np.pi * (X[0]**2+X[1]**2)))


def f_sin(x, y):
    return np.sin(np.pi*x) * np.sin(np.pi*y)

def f_cos(x, y):
    return np.cos(np.pi*x) * np.cos(np.pi*y)

def lap_sin(x, y):
    R = x**2 + y**2

    c = np.cos(2*np.pi*R)
    s = np.sin(2*np.pi*R)

    D = 1 + 16*R*c**2

    term1 = (
        -np.pi**2 * (2 + 16*R*c**2) / D
        * f_sin(x, y)
    )

    term2 = (
        -32*np.pi**2 * x*y*c**2 / D
        * f_cos(x, y)
    )

    bracket = (
        x*np.cos(np.pi*x)*np.sin(np.pi*y)
        + y*np.sin(np.pi*x)*np.cos(np.pi*y)
    )

    term3 = (
        -32*np.pi*c*(c*(1 + 8*R*c**2) - 2*np.pi*R*s) / D**2
        * bracket
    )

    return term1 + term2 + term3

In [ ]:

%%script false --no-raise-error

#source term
def f(x,y,z):
  #return -2*(x**2+y**2-2)
  #return 2 * np.pi**2 * np.sin(np.pi*x)*np.sin(np.pi*y)
  return -1 * lap_franke(x,y)


for ITERATION in [0,1,2,3,4]:


  vertices, quads = make_structured_quad_mesh(-1, 1, -1, 1, 3, 3)

  for i in range(ITERATION):
    vertices, quads = subdivide_quad_mesh(vertices, quads)

  #upgrade to quadratic mesh
  Vertices,Faces = make_bicubic_mesh(vertices, quads)

  N = len(Vertices)

  #Initialize System
  A = np.zeros((N,N))
  b = np.zeros(N)

  start_time = time.perf_counter()

  #Load System
  create_global_stiffness(A,Vertices,Faces)
  create_global_load(b,f,Vertices,Faces)

  #Get Bounds
  boundary_vertices,boundary_edges = get_Boundary(Faces)


  #Define Boundary Conditions
  def g(x,y,z):
    #return 0
    return franke(x,y)

  def h(x,y,z):
    return 0

  def alpha(x,y,z):
    return 1

  def r(x,y,z):
    return 0

  #Apply Boundary Conditions

  apply_dirichlet(A,b,Vertices,boundary_vertices,g)
  #apply_neumann(A,b,Vertices,boundary_edges,h)
  #apply_robin(A,b,Vertices,boundary_edges,alpha,r)

  #Solve
  U = np.linalg.solve(A,b)

  end_time = time.perf_counter()

  #Save Results
  File = open(f'./solutions/planar-franke-O3/{ITERATION}.csv','w')
  File.write('x,y,u\n')

  print(f"Vertices: {len(U)},  Runtime: {end_time - start_time:.6f} seconds")

  boundary_set = set(boundary_vertices)

  with tqdm.tqdm(total=N, desc="Writing results to file") as pbar:
    for i in range(N):
        (x, y, z) = Vertices[i]
        if i in boundary_set:
            File.write(f"{x}, {y}, {U[i]}, boundary\n")
        else:
            File.write(f"{x}, {y}, {U[i]}\n")
        pbar.update(1)

    File.close()


In [ ]:

#source term
def f(x,y,z):
  #return -2*(x**2+y**2-2)
  #return 2 * np.pi**2 * np.sin(np.pi*x)*np.sin(np.pi*y)
    D = 1.0 + 4.0*x**2 + 4.0*y**2

    term1 = 8.0*x**4 + 8.0*y**4 + 2.0*x**2 + 2.0*y**2 - 4.0 - 48.0*x**2*y**2

    term2 = 16.0*x**2*y**2 - 8.0*x**2 - 8.0*y**2

    return -term1 / D + term2 / D**2

  #return -1 * lap_sin(x,y)


for ITERATION in [0,1,2,3]:


  vertices, quads = make_structured_quad_mesh(-1, 1, -1, 1, 3, 3)

  for i in range(ITERATION):
    vertices, quads = subdivide_quad_mesh(vertices, quads)

  #upgrade to quadratic mesh
  Vertices,Faces = make_bicubic_mesh(vertices, quads, project=surface)

  N = len(Vertices)

  #Initialize System
  A = np.zeros((N,N))
  b = np.zeros(N)

  start_time = time.perf_counter()

  #Load System
  create_global_stiffness(A,Vertices,Faces)
  create_global_load(b,f,Vertices,Faces)

  #Get Bounds
  boundary_vertices,boundary_edges = get_Boundary(Faces)


  #Define Boundary Conditions
  def g(x,y,z):
    #return 0
    return 0

  def h(x,y,z):
    return 0

  def alpha(x,y,z):
    return 1

  def r(x,y,z):
    return 0

  #Apply Boundary Conditions

  apply_dirichlet(A,b,Vertices,boundary_vertices,g)
  #apply_neumann(A,b,Vertices,boundary_edges,h)
  #apply_robin(A,b,Vertices,boundary_edges,alpha,r)

  #Solve
  U = np.linalg.solve(A,b)

  end_time = time.perf_counter()

  #Save Results
  File = open(f'./solutions/parab-quad-O3/{ITERATION}.csv','w')
  File.write('x,y,u\n')

  print(f"Vertices: {len(U)},  Runtime: {end_time - start_time:.6f} seconds")

  boundary_set = set(boundary_vertices)

  with tqdm.tqdm(total=N, desc="Writing results to file") as pbar:
    for i in range(N):
        (x, y, z) = Vertices[i]
        if i in boundary_set:
            File.write(f"{x}, {y}, {U[i]}, boundary\n")
        else:
            File.write(f"{x}, {y}, {U[i]}\n")
        pbar.update(1)

    File.close()
